In [1]:
# installing required libraries
%pip install transformers datasets accelerate -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
# imports and setup

import os
import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# data loading and supercategory mapping
BASE_DIR = ".."
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")

df_train = pd.read_csv(os.path.join(PROCESSED_DIR, "train.csv"))
df_val = pd.read_csv(os.path.join(PROCESSED_DIR, "val.csv"))
df_test = pd.read_csv(os.path.join(PROCESSED_DIR, "test.csv"))

# подгружаем supercategory mapping
mapping = pd.read_csv(os.path.join(PROCESSED_DIR, "label_to_supercategory_v1.csv"))
label_to_supercat = dict(zip(mapping["label"], mapping["supercategory"]))

for df in [df_train, df_val, df_test]:
    df["supercategory"] = df["label"].map(label_to_supercat)

print(df_train["supercategory"].value_counts())

supercategory
sysadmin_devops_network     3308
generic_it_ops              2913
it_governance_leadership    2129
project_product             2006
technical_specialized       1982
backend_general_dev         1546
sales_account               1029
tech_support_helpdesk       1016
web_frontend                 601
Name: count, dtype: int64


In [4]:
# label encoding for supercategory

le = LabelEncoder()

df_train["y"] = le.fit_transform(df_train["supercategory"])
df_val["y"] = le.transform(df_val["supercategory"])
df_test["y"] = le.transform(df_test["supercategory"])

num_labels = len(le.classes_)
print("Num labels:", num_labels)

Num labels: 9


In [5]:
# load base bert model
from transformers import AutoModel

AutoModel.from_pretrained("bert-base-uncased")

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [6]:
# tokenizer setup
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [7]:
# tokenize resume texts
def tokenize(batch):
    return tokenizer(
        batch["resume_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [8]:
# create torch dataset objects

train_dataset = Dataset.from_pandas(df_train[["resume_text", "y"]])
val_dataset = Dataset.from_pandas(df_val[["resume_text", "y"]])
test_dataset = Dataset.from_pandas(df_test[["resume_text", "y"]])

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("y", "labels")
val_dataset = val_dataset.rename_column("y", "labels")
test_dataset = test_dataset.rename_column("y", "labels")

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map: 100%|██████████| 5510/5510 [00:07<00:00, 706.59 examples/s]


In [9]:
# load bert for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
# evaluation metrics for validation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

In [11]:
# groupDRO preparation

# create numerical city ids (ONLY for train)
city_to_id = {c:i for i,c in enumerate(df_train["city_group"].astype(str).unique())}

df_train_gdro = df_train.copy()
df_val_gdro   = df_val.copy()
df_test_gdro  = df_test.copy()

df_train_gdro["city_id"] = df_train_gdro["city_group"].astype(str).map(city_to_id).astype(int)
df_val_gdro["city_id"]   = df_val_gdro["city_group"].astype(str).map(city_to_id).fillna(-1).astype(int)
df_test_gdro["city_id"]  = df_test_gdro["city_group"].astype(str).map(city_to_id).fillna(-1).astype(int)

print("Number of city groups:", len(city_to_id))

Number of city groups: 41


In [12]:
# build dataset with city_id

train_dataset_gdro = Dataset.from_pandas(df_train_gdro[["resume_text","y","city_id"]])
val_dataset_gdro   = Dataset.from_pandas(df_val_gdro[["resume_text","y","city_id"]])
test_dataset_gdro  = Dataset.from_pandas(df_test_gdro[["resume_text","y","city_id"]])

train_dataset_gdro = train_dataset_gdro.map(tokenize, batched=True)
val_dataset_gdro   = val_dataset_gdro.map(tokenize, batched=True)
test_dataset_gdro  = test_dataset_gdro.map(tokenize, batched=True)

train_dataset_gdro = train_dataset_gdro.rename_column("y", "labels")
val_dataset_gdro   = val_dataset_gdro.rename_column("y", "labels")
test_dataset_gdro  = test_dataset_gdro.rename_column("y", "labels")

train_dataset_gdro.set_format("torch", columns=["input_ids","attention_mask","labels","city_id"])
val_dataset_gdro.set_format("torch", columns=["input_ids","attention_mask","labels","city_id"])
test_dataset_gdro.set_format("torch", columns=["input_ids","attention_mask","labels","city_id"])

print(train_dataset_gdro[0].keys())

Map: 100%|██████████| 5510/5510 [00:07<00:00, 750.77 examples/s]

dict_keys(['labels', 'city_id', 'input_ids', 'attention_mask'])


In [13]:
# groupdro trainer class

import torch
from transformers import Trainer

class GroupDROTrainer(Trainer):
    def __init__(self, *args, num_groups: int, eta: float = 0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_groups = num_groups
        self.eta = eta
        self.q = torch.ones(num_groups) / num_groups

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        city_id = inputs.pop("city_id")
        labels = inputs.get("labels")

        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
        per_sample_loss = loss_fct(logits, labels)

        group_loss = torch.zeros(self.num_groups, device=per_sample_loss.device)
        group_present = torch.zeros(self.num_groups, device=per_sample_loss.device)

        for g in range(self.num_groups):
            mask = (city_id == g)
            if mask.any():
                group_loss[g] = per_sample_loss[mask].mean()
                group_present[g] = 1.0

        with torch.no_grad():
            q = self.q.to(per_sample_loss.device)
            q = q * torch.exp(self.eta * group_loss * group_present)
            q = q / q.sum()
            self.q = q.detach().cpu()

        q = self.q.to(per_sample_loss.device)
        loss = (q * group_loss).sum()

        return (loss, outputs) if return_outputs else loss

In [14]:
# groupdro training arguments

from transformers import TrainingArguments

training_args_gdro = TrainingArguments(
    output_dir="./bert_9classes_gdro",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=100,
    remove_unused_columns=False,   # важно
)

In [15]:
# train groupDRO model

num_groups = len(city_to_id)

model_gdro = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

trainer_gdro = GroupDROTrainer(
    model=model_gdro,
    args=training_args_gdro,
    train_dataset=train_dataset_gdro,
    eval_dataset=val_dataset_gdro,
    compute_metrics=compute_metrics,
    num_groups=num_groups,
    eta=0.1
)

trainer_gdro.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
100,1.784800
200,2.014400
300,2.002200
400,2.064000
500,2.028600
600,1.939500
700,1.966400
800,1.815400
900,1.811500
1000,1.562500


TrainOutput(global_step=2067, training_loss=1.6608719292849692, metrics={'train_runtime': 2215.1119, 'train_samples_per_second': 7.462, 'train_steps_per_second': 0.933, 'total_flos': 1087374773675520.0, 'train_loss': 1.6608719292849692, 'epoch': 1.0})

In [16]:
# groupdro evaluation

predictions_gdro = trainer_gdro.predict(test_dataset_gdro)

y_true_gdro = predictions_gdro.label_ids
y_pred_gdro = np.argmax(predictions_gdro.predictions, axis=1)

print("GroupDRO Accuracy:", accuracy_score(y_true_gdro, y_pred_gdro))
print("GroupDRO Macro F1:", f1_score(y_true_gdro, y_pred_gdro, average="macro"))

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


GroupDRO Accuracy: 0.4662431941923775
GroupDRO Macro F1: 0.45083816674004545


In [19]:
# SAVE ARTIFACTS: BERT GroupDRO (model_gdro)

from pathlib import Path
import json
import joblib

REPO_ROOT = Path.cwd()
ART_DIR = REPO_ROOT / "models" / "bert_9classes_gdro_eta0.1_1ep"
ART_DIR.mkdir(parents=True, exist_ok=True)
print("Saving to:", ART_DIR)

FINAL_MODEL = model_gdro        # из training cell
FINAL_TOKENIZER = tokenizer

FINAL_MODEL.save_pretrained(ART_DIR, safe_serialization=True)
FINAL_TOKENIZER.save_pretrained(ART_DIR)

classes = list(le.classes_)
label2id = {label: int(i) for i, label in enumerate(classes)}
id2label = {int(i): label for i, label in enumerate(classes)}

with open(ART_DIR / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, ensure_ascii=False, indent=2)

joblib.dump(le, ART_DIR / "label_encoder.joblib")

print("DONE.")

Saving to: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/models/bert_9classes_gdro_eta0.1_1ep
DONE.


In [20]:
# robust fairness (city n >= 50)

def filter_groups_by_min_n(df, group_col, min_n=50):
    counts = df[group_col].value_counts(dropna=False)
    keep = counts[counts >= min_n].index
    df_f = df[df[group_col].isin(keep)].copy()
    return df_f, counts, keep

In [21]:
# fairness helper functions (gap/tpr/fpr)

import numpy as np
import pandas as pd


def gap_by_group(df, group_col):
    """
    Accuracy gap:
    max_group_accuracy - min_group_accuracy
    """
    group_acc = (
        df.groupby(group_col)["correct"]
        .mean()
        .dropna()
    )
    
    if len(group_acc) == 0:
        return np.nan
    
    return group_acc.max() - group_acc.min()


def ovr_rates_by_group(df, group_col, num_classes):
    """
    Computes one-vs-rest TPR and FPR for each class and group.

    Returns:
        tpr: array (num_groups, num_classes)
        fpr: array (num_groups, num_classes)
        support_pos: number of positive samples per (group, class)
        support_neg: number of negative samples per (group, class)
    """
    
    groups = sorted(df[group_col].dropna().unique())
    group_index = {g: i for i, g in enumerate(groups)}
    
    tpr = np.zeros((len(groups), num_classes))
    fpr = np.zeros((len(groups), num_classes))
    support_pos = np.zeros((len(groups), num_classes))
    support_neg = np.zeros((len(groups), num_classes))
    
    for g in groups:
        df_g = df[df[group_col] == g]
        gi = group_index[g]
        
        y_true = df_g["y_true"].values
        y_pred = df_g["y_pred"].values
        
        for c in range(num_classes):
            # Positive = class c
            pos_mask = (y_true == c)
            neg_mask = (y_true != c)
            
            TP = np.sum((y_pred == c) & pos_mask)
            FP = np.sum((y_pred == c) & neg_mask)
            FN = np.sum((y_pred != c) & pos_mask)
            TN = np.sum((y_pred != c) & neg_mask)
            
            support_pos[gi, c] = np.sum(pos_mask)
            support_neg[gi, c] = np.sum(neg_mask)
            
            # TPR = TP / (TP + FN)
            if (TP + FN) > 0:
                tpr[gi, c] = TP / (TP + FN)
            else:
                tpr[gi, c] = np.nan
            
            # FPR = FP / (FP + TN)
            if (FP + TN) > 0:
                fpr[gi, c] = FP / (FP + TN)
            else:
                fpr[gi, c] = np.nan
    
    return tpr, fpr, support_pos, support_neg


def summarize_gaps(rate_matrix):
    """
    Given rate matrix (num_groups, num_classes),
    computes:
        - gap per class
        - worst class gap
        - macro gap (mean over classes)
    """
    
    gap_per_class = []
    
    for c in range(rate_matrix.shape[1]):
        col = rate_matrix[:, c]
        col = col[~np.isnan(col)]
        
        if len(col) == 0:
            gap_per_class.append(np.nan)
        else:
            gap_per_class.append(np.max(col) - np.min(col))
    
    gap_per_class = np.array(gap_per_class)
    
    valid = gap_per_class[~np.isnan(gap_per_class)]
    
    if len(valid) == 0:
        return gap_per_class, np.nan, np.nan
    
    worst_gap = np.max(valid)
    macro_gap = np.mean(valid)
    
    return gap_per_class, worst_gap, macro_gap